##### IMPORTS AND FUNCTION DEFINITIONS

In [1]:
import pandas as pd
from pyannote.metrics.diarization import DiarizationErrorRate
from pyannote.core import Segment, Timeline, Annotation
import os

def convert_model_csv_to_rrtm(model_root_folder, csv_filename):
    """
    Convert the diarization model's CSV output to RTTM format for scoring with pyannote diarization error metrics.
    
    Args:
        model_root_folder (str): The root folder where the model's CSV output is located.
        csv_filename (str): The name of the CSV file to convert.
    Returns:
        None: Writes the RTTM file to the same directory as the CSV file in the diarization model's output folder.
    """
    with open(os.path.join(model_root_folder, csv_filename[:-4]+".rttm"), 'w') as rttm_file:
        df = pd.read_csv(f"{model_root_folder}/{csv_filename}")
        for _, row in df.iterrows():
            # if(row['speaker'] == 'speaker_0'):
                begin = float(row['start_time'])
                end = float(row['end_time'])
                speaker = row['speaker']
                # RTTM format: SPEAKER <file> <channel> <begin> <duration> <conf> <speech_type> <speaker_type> <speaker_name>
                duration = end - begin
                rttm_line = f"SPEAKER {begin:.3f} {duration:.3f} <NA> <NA> <NA> {speaker}\n"
                rttm_file.write(rttm_line)

def convert_gold_label_txt_to_rrtm(gold_label_root_folder, txt_filename):
    """
    Convert the gold label diarization annotations from a TXT file (from ELAN) to RTTM format for scoring with pyannote diarization error metrics.
    
    Args:
        gold_label_root_folder (str): The root folder where the gold label TXT file is located.
        txt_filename (str): The name of the TXT file to convert.
    Returns:
        None: Writes the RTTM file to the same directory as the TXT file in the gold label annotations folder.
    """

    df = pd.read_csv(os.path.join(gold_label_root_folder, txt_filename), sep='\t')
    # Create RTTM output
    with open(os.path.join(gold_label_root_folder, txt_filename[:-4]+".rttm"), 'w') as rttm_file:
        for idx, row in df.iterrows():
            # if(row['speaker'] == 'SPEAKER_1'):
                begin = float(row['speech_start'])
                end = float(row['speech_end'])
                speaker = row['speaker']
                # RTTM format: SPEAKER <file> <channel> <begin> <duration> <conf> <speech_type> <speaker_type> <speaker_name>
                duration = end - begin
                rttm_line = f"SPEAKER {begin:.3f} {duration:.3f} <NA> <NA> <NA> {speaker}\n"
                rttm_file.write(rttm_line)

def load_rttm_to_annotation(path):
    """Load an RTTM file and convert it to a pyannote.core.Annotation object.
    Args:
        path (str): Path to the RTTM file.
    Returns:
        pyannote.core.Annotation: The loaded annotation."""
    ann = Annotation()
    with open(path, 'r') as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if parts[0] != 'SPEAKER':
                continue
            # handle both custom and standard RTTM token positions
            try:
                start = float(parts[1])
                dur = float(parts[2])
                speaker = parts[-1]
            except Exception:
                start = float(parts[3])
                dur = float(parts[4])
                speaker = parts[-1]
            ann[Segment(start, start + dur)] = speaker
    return ann

def der_component_conversion(der_dict):
    """Convert the DER components from a dictionary to a more human-readable format.
    Args:
        der_dict (dict): The dictionary containing DER components.
    Returns:
        dict: A dictionary with human-readable DER components."""
    return {
        'missed_speech_percent': f"{100*der_dict['missed detection']/der_dict['total']:.2f}",
        'false_alarm_percent': f"{100*der_dict['false alarm']/der_dict['total']:.2f}",
        'confusion_percent': f"{100*der_dict['confusion']/der_dict['total']:.2f}",
        "correct_percent": f"{100*der_dict['correct']/der_dict['total']:.2f}",
        "der": f"{100*der_dict['diarization error rate']:.2f}"
    }

##### SPECIFY FILE_ID, MODEL_FOLDERS AND RESULT FORMAT

In [2]:
ids = [
    "p5-s8",
    "p5-s10",
    "p7-s8",
    "p9-s3-1",
    "p9-s9",
    "p11-s4",
    "p11-s11",
    "p12-s3",
    "p12-s6",
    "p17-s2",
    "p17-s6",
    "p18-s15",
    "p18-s17",
]

model_folders = [
    # "ClusteringDiarizer_Test9/denoised_audios/fixed_num-speakers/vad_marblenet/titanet_large/pred_rttms/",
    # "ClusteringDiarizer_Test9/denoised_audios/fixed_num-speakers/vad_marblenet/ecapa_tdnn/pred_rttms/",
    "ClusteringDiarizer_Test15/denoised_audios/fixed_num-speakers/vad_multilingual_marblenet/titanet_large/pred_rttms/",
    # "ClusteringDiarizer_Test15/denoised_audios/fixed_num-speakers/vad_multilingual_marblenet/ecapa_tdnn/pred_rttms/",
    # "ClusteringDiarizer_Test1/denoised_audios/fixed_num-speakers/vad_telephony_marblenet/titanet_large/pred_rttms/",
    # "ClusteringDiarizer_Test1/denoised_audios/fixed_num-speakers/vad_telephony_marblenet/ecapa_tdnn/pred_rttms/",
]

results = {
    "file": [],
    "VAD": [],
    "Embedding": [],
    "DER": [],
    "False Alarm %": [],
    "Missed Speech %": [],
    "Confusion %": [],
    "Correct %": []
}

##### LOOP THROUGH COMBINATIONS

In [3]:
ref_root = "gold_label_diarization_annotations"

for model_folder in model_folders:

    hyp_root = os.path.join("outputs", model_folder)

    for file_id in ids:

        ref_file_name = f"{file_id}_gold_labels"
        hyp_file_name = f"{file_id}_denoised"

        try:

            convert_model_csv_to_rrtm(
                hyp_root,
                hyp_file_name + ".csv"
            )

            convert_gold_label_txt_to_rrtm(
                ref_root,
                ref_file_name + ".txt"
            )

            ref = load_rttm_to_annotation(
                os.path.join(
                    ref_root,
                    ref_file_name + ".rttm"
                )
            )

            hyp = load_rttm_to_annotation(
                os.path.join(
                    hyp_root,
                    hyp_file_name + ".rttm"
                )
            )

            metric = DiarizationErrorRate(
                collar=0.25,
                skip_overlap=False
            )

            der = der_component_conversion(
                metric(ref, hyp, detailed=True)
            )

            parts = model_folder.split("/")
            vad_name = parts[3]
            embedding_name = parts[4]

            results["file"].append(file_id)
            results["VAD"].append(vad_name)
            results["Embedding"].append(embedding_name)
            results["DER"].append(float(der["der"]))
            results["False Alarm %"].append(float(der["false_alarm_percent"]))
            results["Missed Speech %"].append(float(der["missed_speech_percent"]))
            results["Confusion %"].append(float(der["confusion_percent"]))
            results["Correct %"].append(float(der["correct_percent"]))

        except Exception as e:

            print(
                f"Failed: {model_folder} | {file_id}"
            )

            print(e)


/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 

##### PRINT OUTPUTS

In [4]:
df = pd.DataFrame(results)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
df

,file,VAD,Embedding,DER,False Alarm %,Missed Speech %,Confusion %,Correct %
0,p5-s8,vad_multilingual_marblenet,titanet_large,31.55,9.53,13.42,8.60,77.98
1,p5-s10,vad_multilingual_marblenet,titanet_large,20.68,8.51,7.40,4.78,87.82
2,p7-s8,vad_multilingual_marblenet,titanet_large,56.45,9.60,17.90,28.95,53.15
3,p9-s3-1,vad_multilingual_marblenet,titanet_large,66.40,8.90,36.70,20.81,42.50
4,p9-s9,vad_multilingual_marblenet,titanet_large,36.01,13.68,14.45,7.87,77.67
5,p11-s4,vad_multilingual_marblenet,titanet_large,29.43,8.42,10.48,10.53,78.99
6,p11-s11,vad_multilingual_marblenet,titanet_large,27.03,10.27,4.34,12.43,83.23
7,p12-s3,vad_multilingual_marblenet,titanet_large,51.88,30.20,9.37,12.31,78.32
8,p12-s6,vad_multilingual_marblenet,titanet_large,62.76,32.74,13.26,16.76,69.98
9,p17-s2,vad_multilingual_marblenet,titanet_large,40.93,6.72,13.03,21.18,65.79


In [5]:
df[[
    "file",
    "VAD",
    "Embedding",
    "DER",
    "False Alarm %",
    "Missed Speech %",
    "Confusion %",
    "Correct %"
]]

,file,VAD,Embedding,DER,False Alarm %,Missed Speech %,Confusion %,Correct %
0,p5-s8,vad_multilingual_marblenet,titanet_large,31.55,9.53,13.42,8.60,77.98
1,p5-s10,vad_multilingual_marblenet,titanet_large,20.68,8.51,7.40,4.78,87.82
2,p7-s8,vad_multilingual_marblenet,titanet_large,56.45,9.60,17.90,28.95,53.15
3,p9-s3-1,vad_multilingual_marblenet,titanet_large,66.40,8.90,36.70,20.81,42.50
4,p9-s9,vad_multilingual_marblenet,titanet_large,36.01,13.68,14.45,7.87,77.67
5,p11-s4,vad_multilingual_marblenet,titanet_large,29.43,8.42,10.48,10.53,78.99
6,p11-s11,vad_multilingual_marblenet,titanet_large,27.03,10.27,4.34,12.43,83.23
7,p12-s3,vad_multilingual_marblenet,titanet_large,51.88,30.20,9.37,12.31,78.32
8,p12-s6,vad_multilingual_marblenet,titanet_large,62.76,32.74,13.26,16.76,69.98
9,p17-s2,vad_multilingual_marblenet,titanet_large,40.93,6.72,13.03,21.18,65.79


In [6]:
summary = (
    df.groupby(["VAD", "Embedding"])
      .agg({
          "DER":"mean",
          "False Alarm %":"mean",
          "Missed Speech %":"mean",
          "Confusion %":"mean"
      })
      .sort_values("DER")
)

summary

,,DER,False Alarm %,Missed Speech %,Confusion %
VAD,Embedding,,,,
vad_multilingual_marblenet,titanet_large,37.763846,11.863077,13.255385,12.648462
